### This notebook inspects the three IP marts:
#### - data/marts/fct_ip_flows_yearly.csv.gz
#### - data/marts/fct_ip_flows_by_class_yearly.csv.gz
#### - data/marts/fct_ip_flows_by_residency.csv.gz

In [ ]:
from pathlib import Path
import os
import pandas as pd

try:
    import duckdb  # fast & memory-efficient for big CSVs
    HAVE_DUCKDB = True
except Exception:
    HAVE_DUCKDB = False

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.max_colwidth", None)  # don't truncate cell values
pd.set_option("display.width", None) 
pd.set_option("display.max_seq_items", None)  # don't abbreviate lists/tuples in cells





In [ ]:
# Locate the files - navigate up from notebooks folder to project root
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_MARTS = BASE_DIR / "data" / "marts"

FACT_YEARLY = DATA_MARTS / "fct_ip_flows_yearly.csv.gz"
FACT_CLASS = DATA_MARTS / "fct_ip_flows_by_class_yearly.csv.gz"
FACT_RESIDENCY = DATA_MARTS / "fct_ip_flows_by_residency.csv.gz"

print("YEARLY exists:", FACT_YEARLY.exists(), "->", FACT_YEARLY)
print("BY_CLASS exists:", FACT_CLASS.exists(), "->", FACT_CLASS)
print("BY_RESIDENCY exists:", FACT_RESIDENCY.exists(), "->", FACT_RESIDENCY)

assert FACT_YEARLY.exists(), f"Missing: {FACT_YEARLY}"
assert FACT_CLASS.exists(), f"Missing: {FACT_CLASS}"
assert FACT_RESIDENCY.exists(), f"Missing: {FACT_RESIDENCY}"

In [ ]:
# helper functions
def _qident(col: str) -> str:
    """Quote a DuckDB identifier safely (minimal)."""
    return '"' + col.replace('"','""') + '"'

def duck_scan(path: Path) -> str:
    """DuckDB scan expression for CSV.gz (no full read)."""
    return f"read_csv_auto('{path.as_posix()}')"

def table_overview(path: Path):
    """Row count and head using DuckDB (fallback: pandas)."""
    if HAVE_DUCKDB:
        scan = duck_scan(path)
        rowcount = duckdb.sql(f"SELECT COUNT(*) FROM {scan}").fetchone()[0]
        head = duckdb.sql(f"SELECT * FROM {scan} LIMIT 5").df()
        schema = duckdb.sql(f"DESCRIBE SELECT * FROM {scan}").df()
        return rowcount, schema, head
    else:
        df = pd.read_csv(path, nrows=5000)
        rowcount = len(pd.read_csv(path, usecols=[0], engine="python"))
        schema = pd.DataFrame({"column": df.columns, "dtype(pandas)": [str(t) for t in df.dtypes]})
        head = df.head(5)
        return rowcount, schema, head

def distinct_report(path: Path, sample_n: int = 25):
    """
    Compute distinct count per column, plus up to `sample_n` example distinct values.
    Uses DuckDB when available (recommended).
    """
    if HAVE_DUCKDB:
        scan = duck_scan(path)
        cols = duckdb.sql(f"SELECT * FROM {scan} LIMIT 0").df().columns.tolist()
        recs = []
        for c in cols:
            qc = _qident(c)
            n_unique = duckdb.sql(f"SELECT COUNT(DISTINCT {qc}) FROM {scan}").fetchone()[0]
            # sample unique values (ordered; text sorts lexicographically)
            sample = duckdb.sql(f"SELECT DISTINCT {qc} AS v FROM {scan} ORDER BY {qc} NULLS FIRST LIMIT {sample_n}").df()["v"].tolist()
            recs.append({"column": c, "n_unique": n_unique, "sample_values": sample})
        return pd.DataFrame(recs).sort_values("column").reset_index(drop=True)
    else:
        # Fallback: pandas (loads whole file, OK for quick looks but slower)
        df = pd.read_csv(path)
        out = []
        for c in df.columns:
            vals = df[c].drop_duplicates()
            out.append({
                "column": c,
                "n_unique": int(vals.shape[0]),
                "sample_values": vals.sort_values(kind="stable").head(sample_n).tolist(),
            })
        return pd.DataFrame(out).sort_values("column").reset_index(drop=True)


In [ ]:
# fct_ip_flows_yearly overview
rows, schema, head = table_overview(FACT_YEARLY)
print(f"Rows (yearly): {rows}")
display(schema)
display(head)


In [ ]:
# distinct values per column (yearly)
uniq_yearly = distinct_report(FACT_YEARLY, sample_n=10000)
display(uniq_yearly)


In [ ]:
# overview of fct_ip_flows_by_class_yearly 
rows_c, schema_c, head_c = table_overview(FACT_CLASS)
print(f"Rows (by_class): {rows_c}")
display(schema_c)
display(head_c)

In [ ]:
# Overview of fct_ip_flows_by_class_yearly
uniq_class = distinct_report(FACT_CLASS, sample_n=1000)
display(uniq_class)


In [ ]:
# fct_ip_flows_residency overview
rows, schema, head = table_overview(FACT_RESIDENCY)
print(f"Rows (yearly): {rows}")
display(schema)
display(head)

In [ ]:
# Overview of fct_ip_flows_by_class_yearly
uniq_res = distinct_report(FACT_RESIDENCY, sample_n=1000)
display(uniq_res)
